# Forecasting Adapter demo



In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt
import einops
from PIL import Image
from timm.models.vision_transformer import PatchEmbed

from forecasting_adapter import VisionTSImageConverter, VisionTSImageReconstructor
from imputation_adapter import VisionTSImputationConverter, VisionTSImputationReconstructor
from understanding_adapter import VisionTSFullSequenceConverter, VisionTSFullSequenceReconstructor


In [ ]:
def _to_json_safe(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")


def show_image(image: torch.Tensor, nvars: int, color_list, save_path: Path) -> None:
    imagenet_mean = np.array([0.5, 0.5, 0.5])
    imagenet_std = np.array([0.5, 0.5, 0.5])

    cur_image = torch.zeros_like(image).cpu()
    height_per_var = image.shape[0] // max(nvars, 1)

    for i in range(nvars):
        cur_color = color_list[i]
        cur_image[i * height_per_var : (i + 1) * height_per_var, :, cur_color] = (
            image[i * height_per_var : (i + 1) * height_per_var, :, cur_color].cpu()
            * imagenet_std[cur_color]
            + imagenet_mean[cur_color]
        ) * 255

    cur_image = torch.clip(cur_image, 0, 255).to(torch.uint8).numpy()
    img_pil = Image.fromarray(cur_image)
    img_pil.save(save_path)


def random_masking(x, mask_ratio, noise):
    """MAE style random masking."""
    n_batch, n_tokens, n_dim = x.shape
    len_keep = int(round(n_tokens * (1 - mask_ratio)))
    ids_shuffle = torch.argsort(noise, dim=1)
    ids_restore = torch.argsort(ids_shuffle, dim=1)
    ids_keep = ids_shuffle[:, :len_keep]
    x_masked = torch.gather(x, 1, ids_keep.unsqueeze(-1).repeat(1, 1, n_dim))
    mask = torch.ones([n_batch, n_tokens], device=x.device)
    mask[:, :len_keep] = 0
    mask = torch.gather(mask, 1, ids_restore)
    return x_masked, mask, ids_restore


def unpatchify(x, patch_size=16):
    batch, length, dim = x.shape
    p = patch_size
    h = w = int(length**0.5)
    x = x.reshape(batch, h, w, p, p, 3)
    x = torch.einsum("bhwpqc->bchpwq", x)
    return x.reshape(batch, 3, h * p, w * p)


def build_series(length: int, nvars: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 50, length)
    data = np.zeros((length, nvars), dtype=np.float32)
    for i in range(nvars):
        data[:, i] = np.sin(t * (i + 1) * 0.2) + 0.1 * rng.standard_normal(length)
    return data


def plot_full_var0(path: Path, title: str, real: np.ndarray, recon: np.ndarray) -> None:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(real[:, 0], label="real_var0")
    ax.plot(recon[:, 0], label="recon_var0", alpha=0.7)
    ax.set_xlabel("time")
    ax.set_ylabel("value")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


In [ ]:
# Parameters
context_len = 336
pred_len = 96
nvars = 3
image_size = 896
freq = "H"
period = 24
seed = 123

output_dir = Path("forecasting_demo")
output_dir.mkdir(parents=True, exist_ok=True)

# Build toy series
seq_len = context_len + pred_len
data = build_series(seq_len, nvars, seed)
context = data[:context_len]
pred = data[context_len:]
full_seq = data

# Convert to image
converter = VisionTSImageConverter(
    image_size=image_size,
    patch_size=16,
    num_patch_input=None,
    color=True,
    seed=seed,
)

sample = converter.convert(context_len, pred_len, context, freq=freq, period=period)
metadata = sample.metadata
(output_dir / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False, default=_to_json_safe),
    encoding="utf-8",
)

color_list = metadata.get("color_list")
if color_list is None:
    color_list = [0] * nvars
else:
    color_list = list(np.asarray(color_list, dtype=int))

image_tensor = torch.as_tensor(sample.image).float()
if image_tensor.dim() == 3:
    image_tensor = image_tensor.unsqueeze(0)

patch_size = int(metadata.get("patch_size", 16))
patch_embed_layer = PatchEmbed(image_size, patch_size, 3, embed_dim=1024)
patches = patch_embed_layer(image_tensor)

if "mask" in metadata:
    noise = einops.repeat(
        torch.as_tensor(metadata["mask"]),
        "1 l -> n l",
        n=image_tensor.shape[0],
    )
else:
    noise = torch.rand(image_tensor.shape[0], patches.shape[1])

mask_ratio = float(metadata.get("mask_ratio", 0.0))
_, mask, _ = random_masking(patches, mask_ratio, noise)
mask_expand = mask.unsqueeze(-1).repeat(1, 1, patch_size * patch_size * 3)
mask_img = unpatchify(mask_expand, patch_size)

black_bg = -torch.ones_like(image_tensor) * 2
image_vis = image_tensor * (1 - mask_img) + black_bg * mask_img
image_vis_hwc = image_vis[0].permute(1, 2, 0)

show_image(image_vis_hwc, nvars, color_list, output_dir / "image_context.png")

full_image = converter.render_full_groundtruth_image(
    torch.as_tensor(full_seq),
    sample,
)
full_image_tensor = full_image[0].permute(1, 2, 0)
image_full_path = output_dir / "image_full.png"
show_image(full_image_tensor, nvars, color_list, image_full_path)


In [ ]:
# Reconstruct and evaluate
reconstructor = VisionTSImageReconstructor()
ctx_hat, pred_hat = reconstructor.reconstruct(image_full_path, metadata)

mse_ctx = float(np.mean((context - ctx_hat) ** 2))
mse_pred = float(np.mean((pred - pred_hat) ** 2))
mae_ctx = float(np.mean(np.abs(context - ctx_hat)))
mae_pred = float(np.mean(np.abs(pred - pred_hat)))


def plot_var0(path: Path, title: str) -> None:
    fig, ax = plt.subplots(figsize=(12, 4))
    full_real = np.concatenate([context[:, 0], pred[:, 0]], axis=0)
    full_recon = np.concatenate([ctx_hat[:, 0], pred_hat[:, 0]], axis=0)
    ax.plot(full_real, label="real_var0")
    ax.plot(full_recon, label="recon_var0", alpha=0.7)
    ax.axvline(context_len, color="red", linestyle="--", linewidth=1, label="pred_start")
    ax.set_xlabel("time")
    ax.set_ylabel("value")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)

results = {
    "mse_context": mse_ctx,
    "mse_pred": mse_pred,
    "mae_context": mae_ctx,
    "mae_pred": mae_pred,
    "context_len": context_len,
    "pred_len": pred_len,
    "nvars": nvars,
}
(output_dir / "results.json").write_text(
    json.dumps(results, indent=2, ensure_ascii=False, default=_to_json_safe),
    encoding="utf-8",
)

plot_path = output_dir / "var0_compare.png"
plot_var0(plot_path, "Var0: real vs reconstructed (full sequence)")

print("Saved:")
print(f"- {output_dir / 'image_context.png'}")
print(f"- {output_dir / 'image_full.png'}")
print(f"- {output_dir / 'metadata.json'}")
print(f"- {output_dir / 'results.json'}")
print(f"- {plot_path}")
print("MSE (context):", mse_ctx)
print("MSE (pred):", mse_pred)
print("MAE (context):", mae_ctx)
print("MAE (pred):", mae_pred)


# Imputation adapter demo


In [ ]:
# Imputation adapter demo
imp_image_size = 896
imp_patch_size = 16
imp_nvars = 3
imp_freq = "H"
imp_period = 24
imp_seq_len = 480
imp_seed = 123

imp_output_dir = Path("imputation_demo")
imp_output_dir.mkdir(parents=True, exist_ok=True)

imp_data = build_series(imp_seq_len, imp_nvars, imp_seed)

imp_converter = VisionTSImputationConverter(
    image_size=imp_image_size,
    patch_size=imp_patch_size,
    color=True,
    seed=imp_seed,
    mask_ratio=(0.1, 0.4),
    mask_prob=1.0,
    mask_mode="contiguous",
    mask_value=-1.0,
    store_mask=True,
)

imp_sample = imp_converter.convert(
    imp_data,
    freq=imp_freq,
    period=imp_period,
    apply_mask=True,
)

imp_metadata = imp_sample.metadata
(imp_output_dir / "metadata.json").write_text(
    json.dumps(imp_metadata, indent=2, ensure_ascii=False, default=_to_json_safe),
    encoding="utf-8",
)

imp_color_list = imp_metadata.get("color_list")
if imp_color_list is None:
    imp_color_list = [0] * imp_nvars
else:
    imp_color_list = list(np.asarray(imp_color_list, dtype=int))

imp_masked_tensor = torch.as_tensor(imp_sample.image).float()
imp_masked_hwc = imp_masked_tensor[0].permute(1, 2, 0)
imp_masked_path = imp_output_dir / "image_masked.png"
show_image(imp_masked_hwc, imp_nvars, imp_color_list, imp_masked_path)

imp_full_sample = imp_converter.convert_full_with_stats(
    imp_data,
    freq=imp_freq,
    period=imp_period,
    stats_metadata=imp_metadata,
)
imp_full_tensor = torch.as_tensor(imp_full_sample.image).float()
imp_full_hwc = imp_full_tensor[0].permute(1, 2, 0)
imp_full_path = imp_output_dir / "image_full.png"
show_image(imp_full_hwc, imp_nvars, imp_color_list, imp_full_path)

imp_reconstructor = VisionTSImputationReconstructor()
imp_recon = imp_reconstructor.reconstruct(imp_full_path, imp_metadata)

imp_mse = float(np.mean((imp_data - imp_recon) ** 2))
imp_mae = float(np.mean(np.abs(imp_data - imp_recon)))

mask = imp_metadata.get("mask")
imp_mse_masked = None
imp_mse_unmasked = None
if mask is not None:
    mask_arr = np.asarray(mask).astype(bool)
    data_t = imp_data.T
    recon_t = imp_recon.T
    if mask_arr.any():
        imp_mse_masked = float(np.mean((data_t[mask_arr] - recon_t[mask_arr]) ** 2))
    if (~mask_arr).any():
        imp_mse_unmasked = float(np.mean((data_t[~mask_arr] - recon_t[~mask_arr]) ** 2))

imp_plot_path = imp_output_dir / "var0_compare.png"
plot_full_var0(imp_plot_path, "Imputation: var0 real vs reconstructed", imp_data, imp_recon)

imp_results = {
    "mse": imp_mse,
    "mae": imp_mae,
    "mse_masked": imp_mse_masked,
    "mse_unmasked": imp_mse_unmasked,
    "seq_len": imp_seq_len,
    "nvars": imp_nvars,
    "mask_ratio_actual": float(imp_metadata.get("mask_ratio_actual", 0.0)),
}
(imp_output_dir / "results.json").write_text(
    json.dumps(imp_results, indent=2, ensure_ascii=False, default=_to_json_safe),
    encoding="utf-8",
)

print("Imputation saved:")
print(f"- {imp_masked_path}")
print(f"- {imp_full_path}")
print(f"- {imp_output_dir / 'metadata.json'}")
print(f"- {imp_output_dir / 'results.json'}")
print(f"- {imp_plot_path}")
print("Imputation MSE:", imp_mse)
print("Imputation MAE:", imp_mae)
if imp_mse_masked is not None:
    print("Imputation MSE (masked):", imp_mse_masked)
if imp_mse_unmasked is not None:
    print("Imputation MSE (unmasked):", imp_mse_unmasked)


# Understanding adapter demo


In [ ]:
# Understanding (full sequence) adapter demo
under_image_size = 896
under_patch_size = 16
under_nvars = 3
under_freq = "H"
under_period = 24
under_seq_len = 480
under_seed = 456

under_output_dir = Path("understanding_demo")
under_output_dir.mkdir(parents=True, exist_ok=True)

under_data = build_series(under_seq_len, under_nvars, under_seed)

under_converter = VisionTSFullSequenceConverter(
    image_size=under_image_size,
    patch_size=under_patch_size,
    color=True,
    seed=under_seed,
)

under_sample = under_converter.convert(under_data, freq=under_freq, period=under_period)
under_metadata = under_sample.metadata
(under_output_dir / "metadata.json").write_text(
    json.dumps(under_metadata, indent=2, ensure_ascii=False, default=_to_json_safe),
    encoding="utf-8",
)

under_color_list = under_metadata.get("color_list")
if under_color_list is None:
    under_color_list = [0] * under_nvars
else:
    under_color_list = list(np.asarray(under_color_list, dtype=int))

under_image_tensor = torch.as_tensor(under_sample.image).float()
under_image_hwc = under_image_tensor[0].permute(1, 2, 0)
under_image_path = under_output_dir / "image_full.png"
show_image(under_image_hwc, under_nvars, under_color_list, under_image_path)

under_reconstructor = VisionTSFullSequenceReconstructor()
under_recon = under_reconstructor.reconstruct(under_image_path, under_metadata)

under_mse = float(np.mean((under_data - under_recon) ** 2))
under_mae = float(np.mean(np.abs(under_data - under_recon)))

under_plot_path = under_output_dir / "var0_compare.png"
plot_full_var0(under_plot_path, "Understanding: var0 real vs reconstructed", under_data, under_recon)

under_results = {
    "mse": under_mse,
    "mae": under_mae,
    "seq_len": under_seq_len,
    "nvars": under_nvars,
}
(under_output_dir / "results.json").write_text(
    json.dumps(under_results, indent=2, ensure_ascii=False, default=_to_json_safe),
    encoding="utf-8",
)

print("Understanding saved:")
print(f"- {under_image_path}")
print(f"- {under_output_dir / 'metadata.json'}")
print(f"- {under_output_dir / 'results.json'}")
print(f"- {under_plot_path}")
print("Understanding MSE:", under_mse)
print("Understanding MAE:", under_mae)
